<a href="https://colab.research.google.com/github/diwashsapkota/EMDC-Training-Contents/blob/main/P3_ebook_article_blog_vlog_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standalone Ebook + Content + Chat Generator EN

Derivative JSON + cleaned transcript TXT -> English e-book chapter, articles, blogs, vlog script, and dataset-based chat.

Editorial direction: thoughtful Christian sensibility, but not strict, branded, or movement-based.

## Step 1: Installation

### Step 1.1: API Key Explanation
To install dependencies and run this notebook, you will need an OpenAI API key. The key will be requested in the next step. This key will be used for all requests to the Gemini model.

In [ ]:
# @title Step 1.2: Install dependencies
!pip -q install openai python-docx reportlab

import os
from getpass import getpass

try:
    from google.colab import userdata
    saved_key = userdata.get('OPENAI_API_KEY')
except Exception:
    saved_key = None

OPENAI_API_KEY = saved_key or getpass('Paste your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

if not os.environ['OPENAI_API_KEY'].startswith('sk-'):
    raise ValueError('OPENAI_API_KEY does not look valid.')

print('API key is ready.')

Paste your OpenAI API key: ··········
API key is ready.


In [ ]:
# @title Step 1.3: Global settings
MODEL = 'gpt-5.4' # @param {"type":"string"}
CHAT_MODEL = 'gpt-5-mini' # @param {"type":"string"}
MIN_WORDS = 3500 # @param {"type":"integer"}
PROJECT_NAME = 'ebook_content_chat_en' # @param {"type":"string"}

OUTPUT_FORMAT_TXT = True # @param {"type":"boolean"}
OUTPUT_FORMAT_DOCX = True # @param {"type":"boolean"}
OUTPUT_FORMAT_PDF = True # @param {"type":"boolean"}

from pathlib import Path
import re
import json
from openai import OpenAI

client = OpenAI()

def safe_name(value: str, fallback: str = 'ebook_content_chat_en') -> str:
    value = (value or '').strip() or fallback
    value = re.sub(r'[^A-Za-z0-9._-]+', '_', value)
    return value.strip('._-') or fallback

PROJECT_NAME = safe_name(PROJECT_NAME)
OUT_DIR = Path('/content') / PROJECT_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output folder: {OUT_DIR}')

Output folder: /content/ebook_content_chat_en


In [ ]:
# @title Step 1.4: Shared helper functions
from typing import Any
from pathlib import Path
from html import escape
import zipfile
import json
import re

REQUEST_FALLBACK_KEYS = ['reasoning', 'verbosity', 'store']

def create_response_with_fallback(**kwargs):
    try:
        return client.responses.create(**kwargs)
    except Exception as exc:
        message = str(exc)
        reduced_kwargs = dict(kwargs)
        removed = False
        for key in REQUEST_FALLBACK_KEYS:
            if key in reduced_kwargs and (key in message or isinstance(exc, TypeError)):
                reduced_kwargs.pop(key, None)
                removed = True
        if not removed:
            raise
        return client.responses.create(**reduced_kwargs)

def add_gpt5_options(kwargs: dict, model: str, effort: str = 'medium') -> dict:
    if model.strip().lower().startswith('gpt-5'):
        kwargs['reasoning'] = {'effort': effort}
        kwargs['verbosity'] = 'medium'
        kwargs['store'] = False
    return kwargs

def ensure_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def first_non_empty(*values: Any) -> str:
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if text:
            return text
    return ''

def find_source_link(data: dict) -> str:
    candidates = []
    if isinstance(data.get('sources'), dict):
        candidates.extend([data['sources'].get('youtube_url'), data['sources'].get('url'), data['sources'].get('source_url')])
    derivative = data.get('derivative') if isinstance(data.get('derivative'), dict) else {}
    if isinstance(derivative.get('sources'), dict):
        candidates.extend([derivative['sources'].get('youtube_url'), derivative['sources'].get('url'), derivative['sources'].get('source_url')])
    for item in candidates:
        if item and str(item).strip():
            return str(item).strip()
    return ''

def compact_dataset_context(data: dict, transcript: str, max_chars: int = 30000) -> str:
    source = {
        'about_video': data.get('about_video', {}),
        'derivative': data.get('derivative', {}),
        'source_link': find_source_link(data),
        'transcript_excerpt': transcript[:12000],
    }
    text = json.dumps(source, ensure_ascii=False, indent=2)
    return text[:max_chars]

def full_html_document(title: str, body: str) -> str:
    return f'''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{escape(title)}</title>
</head>
<body>
{body}
</body>
</html>
'''.strip() + '\n'

def zip_folder(folder: Path, zip_path: Path) -> Path:
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in folder.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(folder))
    return zip_path

def choose_writable_output_path(path: Path) -> Path:
    if not path.exists():
        return path
    counter = 1
    while True:
        candidate = path.with_name(f'{path.stem}_new_{counter}{path.suffix}')
        if not candidate.exists():
            return candidate
        counter += 1

## Step 2: Upload Data

### Step 2.1: Upload Input Files
In the next step, you will be prompted to upload two files sequentially: first the derivative JSON file, then the cleaned transcript TXT file. Please ensure you upload both files when prompted.

In [ ]:
# @title Step 2.2: Upload derivative JSON and transcript TXT
from google.colab import files
from pathlib import Path
import json

print('Upload derivative JSON file:')
uploaded_json = files.upload()

if not uploaded_json:
    raise ValueError('No derivative JSON uploaded.')

json_filename = list(uploaded_json.keys())[0]
json_path = Path('/content') / json_filename
data = json.loads(json_path.read_text(encoding='utf-8'))

if not isinstance(data, dict) or 'about_video' not in data or 'derivative' not in data:
    raise ValueError('The uploaded JSON does not look like a derivative JSON file. Expected top-level about_video and derivative keys.')

print(f'Loaded JSON: {json_filename}')
print('Now upload cleaned transcript TXT file:')
uploaded_txt = files.upload()

if not uploaded_txt:
    raise ValueError('No transcript TXT uploaded.')

txt_filename = list(uploaded_txt.keys())[0]
txt_path = Path('/content') / txt_filename
transcript = txt_path.read_text(encoding='utf-8', errors='ignore').strip()

if not transcript:
    raise ValueError('Transcript TXT is empty.')

DATASET_CONTEXT = compact_dataset_context(data, transcript)
print(f'Loaded transcript: {txt_filename}')
print(f'Transcript characters: {len(transcript):,}')
print(f'Dataset context characters: {len(DATASET_CONTEXT):,}')

Upload derivative JSON file:


Saving derivative_en_20230925_AI_Hour_How2AI_Class1284.json to derivative_en_20230925_AI_Hour_How2AI_Class1284 (1).json
Loaded JSON: derivative_en_20230925_AI_Hour_How2AI_Class1284 (1).json
Now upload cleaned transcript TXT file:


Saving 20230925_AI_Hour_How2AI_Class1284.txt to 20230925_AI_Hour_How2AI_Class1284 (1).txt
Loaded transcript: 20230925_AI_Hour_How2AI_Class1284 (1).txt
Transcript characters: 71,095
Dataset context characters: 30,000


# Project

## Step 3: E-Book Generation

In [ ]:
# @title Step 3.1: Build E-book Prompt
def build_outline_text(data: dict[str, Any]) -> str:
    derivative = data.get('derivative', {}) if isinstance(data.get('derivative'), dict) else {}
    chapter = derivative.get('chapter', {}) if isinstance(derivative.get('chapter'), dict) else {}
    outline = ensure_list(chapter.get('outline_chapter'))
    if not outline:
        about_video = data.get('about_video', {}) if isinstance(data.get('about_video'), dict) else {}
        outline = ensure_list(about_video.get('content_outline'))
    rows = []
    for idx, item in enumerate(outline[:8], start=1):
        if isinstance(item, dict):
            title = first_non_empty(item.get('section_title'), item.get('title'), item.get('heading'), f'Section {idx}')
            objective = first_non_empty(item.get('objective'), item.get('summary'), item.get('purpose'))
            key_points = ensure_list(item.get('key_points'))
            point_text = '; '.join(str(point).strip() for point in key_points if str(point).strip())
            line = f'{idx}. {title}'
            if objective:
                line += f' - {objective}'
            if point_text:
                line += f' Key points: {point_text}'
            rows.append(line)
        else:
            text = str(item).strip()
            if text:
                rows.append(f'{idx}. {text}')
    return '\n'.join(rows)

def build_quote_bank(data: dict[str, Any]) -> str:
    about_video = data.get('about_video', {}) if isinstance(data.get('about_video'), dict) else {}
    derivative = data.get('derivative', {}) if isinstance(data.get('derivative'), dict) else {}
    quotes = ensure_list(derivative.get('quotes')) or ensure_list(about_video.get('important_quotes'))
    rows = []
    for item in quotes[:10]:
        if isinstance(item, dict):
            quote = first_non_empty(item.get('quote'))
            meaning = first_non_empty(item.get('meaning'))
            if quote:
                rows.append(f'- "{quote}"' + (f' ({meaning})' if meaning else ''))
        elif str(item).strip():
            rows.append(f'- "{str(item).strip()}"')
    return '\n'.join(rows)

def build_question_block(items: list[Any], keys: list[str], limit: int, fallback_prefix: str) -> str:
    rows = []
    for item in items:
        text = first_non_empty(*(item.get(key) for key in keys)) if isinstance(item, dict) else str(item).strip()
        if text:
            rows.append(f'- {text}')
        if len(rows) >= limit:
            break
    while len(rows) < limit:
        rows.append(f'- {fallback_prefix} {len(rows) + 1}')
    return '\n'.join(rows)

def build_intro_description(data: dict[str, Any]) -> str:
    about_video = data.get('about_video', {}) if isinstance(data.get('about_video'), dict) else {}
    derivative = data.get('derivative', {}) if isinstance(data.get('derivative'), dict) else {}
    parts = [first_non_empty(about_video.get('video_summary_short'), derivative.get('summary')), first_non_empty(about_video.get('core_thesis'), derivative.get('one_sentence_abstract')), first_non_empty(about_video.get('audience_relevance'))]
    return '\n'.join(f'- {part}' for part in parts if part)

def build_ebook_prompt(data: dict[str, Any], transcript: str, min_words: int = 3500) -> str:
    about_video = data.get('about_video', {}) if isinstance(data.get('about_video'), dict) else {}
    derivative_temp = data.get('derivative', {})
    derivative = derivative_temp if isinstance(derivative_temp, dict) else {}
    chapter = derivative.get('chapter', {}) if isinstance(derivative.get('chapter'), dict) else {}
    sources = data.get('sources', {}) if isinstance(data.get('sources'), dict) else {}
    lessons = '\n'.join(f'- {first_non_empty(item.get("point"), item.get("isi"))}: {first_non_empty(item.get("explanation"))}' for item in ensure_list(derivative.get('lessons'))[:15] if isinstance(item, dict) and first_non_empty(item.get('point'), item.get('isi')))
    glossary = '\n'.join(f'- {first_non_empty(item.get("term"), item.get("istilah"))}: {first_non_empty(item.get("definition"), item.get("definisi"))}' for item in (ensure_list(derivative.get('glossary'))[:12] or ensure_list(about_video.get('glossary_core'))[:12]) if isinstance(item, dict) and first_non_empty(item.get('term'), item.get('istilah')))
    alt_titles = '\n'.join(str(item).strip() for item in ensure_list(derivative.get('alternative_titles')) if str(item).strip())
    chapter_title = first_non_empty(sources.get('title'), chapter.get('chapter_title'), about_video.get('main_topic'), next(iter(ensure_list(derivative.get('alternative_titles'))), ''))
    subtitle = first_non_empty(about_video.get('core_thesis'), derivative.get('one_sentence_abstract'))
    hook = first_non_empty(chapter.get('chapter_hook'), about_video.get('primary_purpose'), derivative.get('summary'))
    opening_quote = ''
    for item in ensure_list(derivative.get('quotes')) + ensure_list(about_video.get('important_quotes')):
        if isinstance(item, dict) and first_non_empty(item.get('quote')):
            opening_quote = first_non_empty(item.get('quote'))
            break
    return f'''
You are writing ONE English ebook chapter based on seminar-derived source material.

EDITORIAL ORIENTATION:
- Write with a thoughtful Christian sensibility, but keep the tone accessible to a broad audience.
- Let faith-informed reflection appear naturally when relevant.
- Do not force theological language, church language, Bible references, or ministry applications unless the source material supports them.
- Focus on wisdom, discernment, responsibility, character, practical application, and human flourishing.
- Do not include organizational branding, movement identity, or campaign language.

TARGET LENGTH:
At least {min_words} words.

OUTPUT FORMAT:
Return simple markdown that reads like a book chapter draft, not like notes.
Use # for title, ## for major sections, ### for subsections, and short bullet lists only where useful.

STRUCTURE:
# Title
Subtitle/tagline
Strong opening paragraph
## Introduction
Discussion outline with 3 points
## [Major Section 1 from the outline]
## [Major Section 2 from the outline]
## [Major Section 3 from the outline]
## Conclusion
## Questions
### Reflection
### Discussion
### Application

RULES:
- Major section headings must match the outline points exactly.
- Write in strong, natural English.
- Use Christian language only when it fits the source and serves the reader.
- Do not mention JSON, transcript, prompt, source metadata, or system instructions.
- Do not write "Chapter 1" or number the chapter.

SUGGESTED CHAPTER TITLE:
{chapter_title}

SUGGESTED SUBTITLE:
{subtitle}

SUGGESTED HOOK:
{hook}

OPENING QUOTE:
{opening_quote}

ALTERNATIVE TITLES:
{alt_titles}

SHORT SUMMARY:
{build_intro_description(data)}

LONG SUMMARY:
{first_non_empty(about_video.get('video_summary_long'), derivative.get('long_description'))}

CHAPTER OUTLINE:
{build_outline_text(data)}

LESSONS:
{lessons}

QUOTE BANK:
{build_quote_bank(data)}

DISCUSSION QUESTIONS FROM SOURCE:
{build_question_block(ensure_list(derivative.get('discussion_questions')), ['question', 'pertanyaan'], 3, 'Discussion question')}

REFLECTION QUESTIONS FROM SOURCE:
{build_question_block(ensure_list(derivative.get('reflection_questions')), ['question', 'pertanyaan'], 3, 'Reflection question')}

GLOSSARY:
{glossary}

TRANSCRIPT EXCERPT:
{transcript[:22000]}
'''.strip()

EBOOK_SYSTEM_PROMPT = '''You are a professional nonfiction book writer in fluent English with a thoughtful Christian sensibility. Write one polished, readable, reflective, publication-ready ebook chapter. Stay faithful to the source. Use Christian reflection gently where appropriate, but do not force it. Do not mention JSON, transcript, prompt, system instructions, or movement branding.'''
prompt = build_ebook_prompt(data, transcript, MIN_WORDS)
print(f'E-book prompt characters: {len(prompt):,}')


E-book prompt characters: 30,827


In [ ]:
# @title Step 3.2: Generate E-book Chapter
ebook_request_kwargs = {
    'model': MODEL,
    'input': [{'role': 'system', 'content': EBOOK_SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}],
}
add_gpt5_options(ebook_request_kwargs, MODEL)
response = create_response_with_fallback(**ebook_request_kwargs)
chapter_md = (getattr(response, 'output_text', '') or '').strip()
if not chapter_md:
    raise RuntimeError('Model returned empty output_text.')
print('E-book chapter generated.')
print(f'Characters: {len(chapter_md):,}')
print(chapter_md[:1200])

E-book chapter generated.
Characters: 34,875
# 20230925_AI_Hour_How2AI_Class1284

*AI is already reshaping the world and ministry contexts must learn to use it wisely; the difference between weak and useful AI output is the quality and focus of the prompt.*

> “AI is here, to stay, it’s going to change everything, and we need to deal with it.”

Many people try AI once, ask it a broad question, receive a broad answer, and conclude that the technology is overrated. Yet that first disappointment often reveals less about the tool than about the way we approached it. When we speak vaguely, we usually receive vagueness in return. When we ask with clarity, context, and purpose, the result can become surprisingly helpful. That simple observation sits at the heart of wise AI use. The real issue is not only whether artificial intelligence is powerful, but whether we have learned to guide it responsibly.

## Introduction

The conversation about AI has moved quickly from novelty to necessity. What

In [ ]:
# @title Step 3.3: Save and Download E-book Files
from IPython.display import display, Markdown
from google.colab import files

def render_plaintext_book(markdown_text: str) -> str:
    text = markdown_text.replace('\r\n', '\n')
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.M)
    text = re.sub(r'^[-*]\s+', '- ', text, flags=re.M)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip() + '\n'

def write_docx(markdown_text: str, out_path: Path) -> None:
    from docx import Document
    from docx.shared import Pt
    doc = Document()
    doc.styles['Normal'].font.name = 'Georgia'
    doc.styles['Normal'].font.size = Pt(11)
    for raw_line in markdown_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('# '):
            doc.add_heading(line[2:].strip(), level=1)
        elif line.startswith('## '):
            doc.add_heading(line[3:].strip(), level=2)
        elif line.startswith('### '):
            doc.add_heading(line[4:].strip(), level=3)
        elif line.startswith('- '):
            doc.add_paragraph(line[2:].strip(), style='List Bullet')
        else:
            doc.add_paragraph(line)
    doc.save(out_path)

def write_pdf(markdown_text: str, out_path: Path) -> None:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.units import inch
    from xml.sax.saxutils import escape as xml_escape
    doc = SimpleDocTemplate(str(out_path), pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch, topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='BookBody', parent=styles['BodyText'], fontName='Times-Roman', fontSize=11, leading=16, spaceAfter=8))
    styles.add(ParagraphStyle(name='BookBullet', parent=styles['BookBody'], leftIndent=18, firstLineIndent=-10))
    story = []
    for raw_line in markdown_text.splitlines():
        line = raw_line.strip()
        if not line:
            story.append(Spacer(1, 6)); continue
        if line.startswith('# '):
            story.append(Paragraph(xml_escape(line[2:].strip()), styles['Title']))
        elif line.startswith('## '):
            story.append(Paragraph(xml_escape(line[3:].strip()), styles['Heading2']))
        elif line.startswith('### '):
            story.append(Paragraph(xml_escape(line[4:].strip()), styles['Heading3']))
        elif line.startswith('- '):
            story.append(Paragraph('&#8226; ' + xml_escape(line[2:].strip()), styles['BookBullet']))
        else:
            story.append(Paragraph(xml_escape(line), styles['BookBody']))
    doc.build(story)

# Ensure the output directory exists
ebook_dir = OUT_DIR
ebook_dir.mkdir(parents=True, exist_ok=True) # Ensure it exists, though it should from Step 1.3

ebook_saved_files = []
if OUTPUT_FORMAT_TXT:
    txt_out = choose_writable_output_path(ebook_dir / 'chapter_en.txt')
    txt_out.write_text(render_plaintext_book(chapter_md), encoding='utf-8')
    ebook_saved_files.append(txt_out)
if OUTPUT_FORMAT_DOCX:
    docx_out = choose_writable_output_path(ebook_dir / 'chapter_en.docx')
    write_docx(chapter_md, docx_out)
    ebook_saved_files.append(docx_out)
if OUTPUT_FORMAT_PDF:
    pdf_out = choose_writable_output_path(ebook_dir / 'chapter_en.pdf')
    write_pdf(chapter_md, pdf_out)
    ebook_saved_files.append(pdf_out)

print('Saved e-book files:')
for path in ebook_saved_files:
    print(path)
    files.download(str(path))

display(Markdown(chapter_md[:4000] + ('\n\n...' if len(chapter_md) > 4000 else '')))

Saved e-book files:
/content/ebook_content_chat_en/chapter_en_new_3.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/chapter_en_new_3.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/chapter_en_new_3.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 20230925_AI_Hour_How2AI_Class1284

*AI is already reshaping the world and ministry contexts must learn to use it wisely; the difference between weak and useful AI output is the quality and focus of the prompt.*

> “AI is here, to stay, it’s going to change everything, and we need to deal with it.”

Many people try AI once, ask it a broad question, receive a broad answer, and conclude that the technology is overrated. Yet that first disappointment often reveals less about the tool than about the way we approached it. When we speak vaguely, we usually receive vagueness in return. When we ask with clarity, context, and purpose, the result can become surprisingly helpful. That simple observation sits at the heart of wise AI use. The real issue is not only whether artificial intelligence is powerful, but whether we have learned to guide it responsibly.

## Introduction

The conversation about AI has moved quickly from novelty to necessity. What seemed experimental only recently now touches writing, research, education, administration, communication, translation, and creative work. For many people in service-oriented fields, including ministry, the question is no longer whether AI will matter. It is already affecting the world around us. The pressing question is how to engage it with discernment, competence, and integrity.

This chapter explores that practical challenge through three central ideas:

1. **AI must be addressed because it is already changing the world of work and communication.**
2. **Ministry needs a clear framework for engagement so that learning leads to service rather than fear or passivity.**
3. **The usefulness of AI depends heavily on prompt quality, which can be strengthened through focused, audience-aware communication.**

At one level, this is a chapter about technique. It will discuss prompting, context, structure, and practical examples. But beneath the technique is a deeper concern: stewardship. Any powerful tool asks something of its users. It asks for maturity, honesty, and wisdom. It asks us to think not only about what can be done, but what should be done, for whose benefit, and in what spirit.

Used carelessly, AI can amplify laziness, confusion, and superficiality. Used well, it can support learning, accelerate preparation, organize ideas, clarify communication, and help people serve others more effectively. That is why the conversation matters. The goal is not fascination with technology for its own sake. The goal is to become the kind of people who can use new tools without being used by them.

## AI Must Be Addressed

The first and most basic point is one many people still resist: AI must be addressed. It cannot simply be ignored until it becomes more convenient, more polished, or more familiar. Whether welcomed or feared, it has already arrived as a significant force in modern life.

The seminar’s opening claim is intentionally direct: AI is here, it is here to stay, it is going to change everything, and we need to deal with it. That statement is not meant to create panic. It is meant to cut through denial.

### A present reality, not a distant trend

Some technologies move slowly enough that institutions can postpone their response for years. AI does not appear to be one of them. In particular, conversational AI has advanced so rapidly that it has already begun reshaping habits of work in business, education, media, research, and communication. People are using it to summarize documents, draft materials, generate ideas, revise prose, create training content, brainstorm campaigns, prepare lessons, and handle tasks that once took far longer.

That does not mean AI does all these things perfectly. It does not. It can be inaccurate, bland, overly confident, and occasionally absurd. But imperfect usefulness is still usefulness. A tool does not need to be flawless to become influential. It only needs to become good enough, fast enough, and accessible enough to alter ordinary patterns of work. That t

...

## Step 4: Article Generation

In [ ]:
# @title Step 4.1: Generate Articles
ARTICLE_SCHEMA = {'name':'article_content_en','strict':True,'schema':{'type':'object','additionalProperties':False,'properties':{'articles':{'type':'array','minItems':3,'maxItems':3,'items':{'type':'object','additionalProperties':False,'properties':{'title':{'type':'string'},'angle':{'type':'string'},'html_body':{'type':'string'}},'required':['title','angle','html_body']}}},'required':['articles']}}
ARTICLE_SYSTEM_PROMPT = 'You are a senior English article writer with a thoughtful Christian sensibility. Write faithful, publishable articles from the supplied dataset only. Do not invent facts. Do not mention AI, JSON, transcript, prompt instructions, source metadata, or movement branding. Use Christian reflection gently where appropriate, but do not force it. Return valid JSON only.'
ARTICLE_USER_PROMPT = f'''Generate exactly 3 distinct English articles from the dataset.

Rules:
- Each article must have a distinct title, angle, and structure.
- Target around 1200-1800 words per article.
- Use clean HTML fragments only: h1, h2, h3, p, ul, ol, li, blockquote, strong, em.
- Do not include CSS, markdown, or full HTML documents.
- Make each article clear, developed, practical, reflective, and faithful to the dataset.

DATASET:
{DATASET_CONTEXT}'''
article_kwargs = {'model': MODEL, 'input': [{'role':'system','content':ARTICLE_SYSTEM_PROMPT},{'role':'user','content':ARTICLE_USER_PROMPT}], 'text': {'format': {'type':'json_schema','name':ARTICLE_SCHEMA['name'],'strict':True,'schema':ARTICLE_SCHEMA['schema']}}}
add_gpt5_options(article_kwargs, MODEL)
article_response = create_response_with_fallback(**article_kwargs)
article_assets = json.loads(article_response.output_text)
print('Articles generated:', len(article_assets.get('articles', [])))

Articles generated: 3


In [ ]:
# @title Step 4.2: Save and Download Articles
from google.colab import files
article_dir = OUT_DIR / 'article_en'
article_dir.mkdir(parents=True, exist_ok=True)
article_files = []
for idx, item in enumerate(article_assets.get('articles', []), start=1):
    title = str(item.get('title') or f'Article {idx}').strip()
    body = str(item.get('html_body') or '').strip() or f'<h1>{escape(title)}</h1><p>No content generated.</p>'
    out = article_dir / f'article_{idx}.html'
    out.write_text(full_html_document(title, body), encoding='utf-8')
    article_files.append(out)
article_zip = zip_folder(article_dir, OUT_DIR / 'article_en.zip')
print('Saved article files:')
for path in article_files + [article_zip]:
    print(path)
    files.download(str(path))

Saved article files:
/content/ebook_content_chat_en/article_en/article_1.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/article_en/article_2.html


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/article_en.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Blog Generation

In [ ]:
# @title Step 5.1: Generate Blogs
BLOG_SCHEMA = {'name':'blog_content_en','strict':True,'schema':{'type':'object','additionalProperties':False,'properties':{'blogs':{'type':'array','minItems':3,'maxItems':3,'items':{'type':'object','additionalProperties':False,'properties':{'title':{'type':'string'},'pov':{'type':'string'},'html_body':{'type':'string'}},'required':['title','pov','html_body']}}},'required':['blogs']}}
BLOG_SYSTEM_PROMPT = 'You are a senior English blog writer with a thoughtful Christian sensibility. Write faithful, publishable blogs from the supplied dataset only. Do not invent facts. Do not mention AI, JSON, transcript, prompt instructions, source metadata, or movement branding. Use Christian reflection gently where appropriate, but do not force it. Return valid JSON only.'
BLOG_USER_PROMPT = f'''Generate exactly 3 English blogs from the dataset.

POVs:
1. speaker perspective
2. listener perspective
3. topic perspective

Rules:
- Target around 900-1400 words per blog.
- Use clean HTML fragments only: h1, h2, h3, p, ul, ol, li, blockquote, strong, em.
- Do not include CSS, markdown, or full HTML documents.
- Each blog must feel distinct in voice and framing.
- Keep the writing warm, readable, practical, and faithful to the dataset.

DATASET:
{DATASET_CONTEXT}'''
blog_kwargs = {'model': MODEL, 'input': [{'role':'system','content':BLOG_SYSTEM_PROMPT},{'role':'user','content':BLOG_USER_PROMPT}], 'text': {'format': {'type':'json_schema','name':BLOG_SCHEMA['name'],'strict':True,'schema':BLOG_SCHEMA['schema']}}}
add_gpt5_options(blog_kwargs, MODEL)
blog_response = create_response_with_fallback(**blog_kwargs)
blog_assets = json.loads(blog_response.output_text)
print('Blogs generated:', len(blog_assets.get('blogs', [])))

Blogs generated: 3


In [ ]:
# @title Step 5.2: Save and Download Blogs
from google.colab import files
blog_dir = OUT_DIR / 'blog_en'
blog_dir.mkdir(parents=True, exist_ok=True)
blog_files = []
for idx, item in enumerate(blog_assets.get('blogs', []), start=1):
    title = str(item.get('title') or f'Blog {idx}').strip()
    body = str(item.get('html_body') or '').strip() or f'<h1>{escape(title)}</h1><p>No content generated.</p>'
    out = blog_dir / f'blog_{idx}.html'
    out.write_text(full_html_document(title, body), encoding='utf-8')
    blog_files.append(out)
blog_zip = zip_folder(blog_dir, OUT_DIR / 'blog_en.zip')
print('Saved blog files:')
for path in blog_files + [blog_zip]:
    print(path)
    files.download(str(path))

Saved blog files:
/content/ebook_content_chat_en/blog_en/blog_1.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/blog_en/blog_2.html


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/blog_en.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 6: Vlog Script Generation

In [ ]:
# @title Step 6.1: Generate Vlog Script
VLOG_SCHEMA = {'name':'vlog_script_en','strict':True,'schema':{'type':'object','additionalProperties':False,'properties':{'vlog_script':{'type':'object','additionalProperties':False,'properties':{'title':{'type':'string'},'source_link':{'type':'string'},'script':{'type':'string'}},'required':['title','source_link','script']}},'required':['vlog_script']}}
VLOG_SYSTEM_PROMPT = 'You are a warm English short-form video script writer with a thoughtful Christian sensibility. Write a faithful spoken vlog script from the supplied dataset only. Do not invent facts. Do not mention AI, JSON, transcript, prompt instructions, source metadata, SABDA, SADBA, or movement branding. Do not use SABDA/SADBA greetings, organization names, branding, slogans, or identity language, even if present in the dataset. Use Christian reflection gently where appropriate, but do not force it. Return valid JSON only.'
VLOG_USER_PROMPT = f'''Generate exactly 1 English vlog script from the dataset.

Rules:
- Around 1 minute spoken duration.
- Include title, source_link, and script.
- The script should sound natural when read aloud.
- Use short paragraphs.
- Flow: greeting, hook, central insight, why it matters, practical takeaway, closing invitation.
- Keep the tone warm, clear, and accessible.
- Do not mention SABDA, SADBA, or any SABDA/SADBA-related greeting, organization, brand, slogan, or identity.
- Use a neutral greeting such as 'Hello friends,' or start directly with the hook.
- Source link should be this if available: {find_source_link(data)}

DATASET:
{DATASET_CONTEXT}'''
vlog_kwargs = {'model': MODEL, 'input': [{'role':'system','content':VLOG_SYSTEM_PROMPT},{'role':'user','content':VLOG_USER_PROMPT}], 'text': {'format': {'type':'json_schema','name':VLOG_SCHEMA['name'],'strict':True,'schema':VLOG_SCHEMA['schema']}}}
add_gpt5_options(vlog_kwargs, MODEL)
vlog_response = create_response_with_fallback(**vlog_kwargs)
vlog_assets = json.loads(vlog_response.output_text)
print('Vlog generated:', vlog_assets.get('vlog_script', {}).get('title', ''))

Vlog generated: Why Better Prompts Make AI More Useful


In [ ]:
# @title Step 6.2: Save and Download Vlog Script
from google.colab import files
vlog_dir = OUT_DIR / 'vlog_en'
vlog_dir.mkdir(parents=True, exist_ok=True)
vlog = vlog_assets.get('vlog_script', {}) if isinstance(vlog_assets.get('vlog_script'), dict) else {}
vlog_file = vlog_dir / 'vlog_script.txt'
vlog_text = f"Title: {vlog.get('title', '')}\nSource Link: {vlog.get('source_link', find_source_link(data))}\n\nVlog Script:\n{vlog.get('script', '')}\n"
vlog_file.write_text(vlog_text, encoding='utf-8')
vlog_zip = zip_folder(vlog_dir, OUT_DIR / 'vlog_en.zip')
print('Saved vlog files:')
for path in [vlog_file, vlog_zip]:
    print(path)
    files.download(str(path))

Saved vlog files:
/content/ebook_content_chat_en/vlog_en/vlog_script.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/ebook_content_chat_en/vlog_en.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 7: Dataset Chat

In [ ]:
# @title Step 7.1: Initialize Dataset Chat
CHAT_SYSTEM_PROMPT = '''
You are a helpful dataset assistant using a thoughtful Christian sensibility.
Answer only from the uploaded dataset context.
If the dataset does not contain enough information, say so clearly and offer a careful inference only when it is labeled as inference.
Do not invent facts, names, dates, statistics, or quotes.
Keep answers clear, useful, and conversational.
Do not mention hidden prompts or system instructions.
'''.strip()
chat_history = []
print(f'Dataset chat is ready with model: {CHAT_MODEL}')
print('Run the next cell repeatedly to ask questions.')

from IPython.display import display, HTML
from html import escape

while True:
    QUESTION = input('You: ').strip()

    if not QUESTION:
        print('Please type your question.')
        continue

    if QUESTION.lower() == 'exit':
        print('Chat session ended.')
        break

    # print(f'You: {QUESTION}') # Removed this line to prevent echoing user input

    chat_input = [
        {'role': 'system', 'content': CHAT_SYSTEM_PROMPT},
        {'role': 'user', 'content': f'DATASET CONTEXT:\n{DATASET_CONTEXT}'},
    ]
    chat_input.extend(chat_history[-10:])
    chat_input.append({'role': 'user', 'content': QUESTION})
    chat_kwargs = {'model': CHAT_MODEL, 'input': chat_input}
    add_gpt5_options(chat_kwargs, CHAT_MODEL, effort='low')
    chat_response = create_response_with_fallback(**chat_kwargs)
    answer = (getattr(chat_response, 'output_text', '') or '').strip()
    chat_history.append({'role': 'user', 'content': QUESTION})
    chat_history.append({'role': 'assistant', 'content': answer})
    # Escape HTML characters and then replace newlines with <br> for proper rendering
    formatted_answer = escape(answer).replace('\n', '<br>')
    display(HTML(f'<b>AI:</b> {formatted_answer}'))

Dataset chat is ready with model: gpt-5-mini
Run the next cell repeatedly to ask questions.
You: can you give me the summary?


You: exit
Chat session ended.
